# Chronos-Resonance: Multi-GPU Time Dilation & Grokking Experiment

This notebook is designed to run on Google Colab (or any multi-GPU environment). It executes the **Multi-Now Pulse** as described in the `resone8.md` framework.

### Objective
Generate as many kernels on as many GPUs as possible to independently run a script that outputs the hardware time (`clock64()`) as fast as it can. This data is emitted as a signal to a receiver process, which collects the 'nows' across all GPUs to detect time dilation and Moiré interference patterns.

In [ ]:
import cupy as cp
import numpy as np
import multiprocessing as mp
import time
import matplotlib.pyplot as plt

# Verify GPU availability
num_gpus = cp.cuda.runtime.getDeviceCount()
print(f"Detected {num_gpus} GPU(s) for Chronos-Resonance.")
for i in range(num_gpus):
    props = cp.cuda.runtime.getDeviceProperties(i)
    print(f"GPU {i}: {props['name'].decode('utf-8')}")

In [ ]:
cuda_code = r'''
extern "C" __global__
void record_time_jitter(unsigned long long* times, int num_samples) {
    int tid = blockDim.x * blockIdx.x + threadIdx.x;
    // Rapidly sample the GPU hardware clock (cycle counter)
    for (int i = 0; i < num_samples; i++) {
        times[tid * num_samples + i] = clock64();
    }
}
'''

def gpu_worker(gpu_id, queue, stop_event, threads_per_block=1024, blocks=2048, samples_per_thread=50):
    try:
        cp.cuda.Device(gpu_id).use()
        record_kernel = cp.RawKernel(cuda_code, 'record_time_jitter')
        
        total_threads = threads_per_block * blocks
        buffer_size = total_threads * samples_per_thread
        d_times = cp.zeros(buffer_size, dtype=cp.uint64)
        
        print(f"[GPU {gpu_id}] Emitter Started. Threads: {total_threads}, Samples/Thread: {samples_per_thread}")
        
        while not stop_event.is_set():
            # Fire the 'Now' pulse
            record_kernel((blocks,), (threads_per_block,), (d_times, samples_per_thread))
            cp.cuda.Stream.null.synchronize()
            
            # Retrieve the wave collapse data
            h_times = d_times.get()
            
            # Emit signal to receiver (we send a subset to prevent IPC bottleneck)
            # We capture the first 10,000 thread clock sequences for Moiré analysis
            queue.put((gpu_id, time.time(), h_times[:10000].copy()))
            
    except Exception as e:
        print(f"[GPU {gpu_id}] Error: {e}")

def receiver_worker(queue, stop_event, output_file="time_dilation_data.npy"):
    print("[Receiver] Process Started. Listening for 'Now' signals across the manifold...")
    all_data = []
    
    while not stop_event.is_set() or not queue.empty():
        try:
            gpu_id, host_time, data = queue.get(timeout=0.5)
            all_data.append((gpu_id, host_time, data))
            
            if len(all_data) % 50 == 0:
                print(f"[Receiver] Accumulated {len(all_data)} temporal wave packets...")
        except mp.queues.Empty:
            continue
            
    print("[Receiver] Singularity reached. Saving temporal data...")
    np.save(output_file, np.array(all_data, dtype=object))
    print(f"[Receiver] Data saved to {output_file}")

In [ ]:
# --- EXECUTION CONTROL ---
DURATION_SECONDS = 10  # How long to run the experiment

if __name__ == '__main__':
    # Set start method to spawn for CUDA compatibility in multiprocessing
    try:
        mp.set_start_method('spawn')
    except RuntimeError:
        pass # Already set
        
    queue = mp.Queue()
    stop_event = mp.Event()
    
    # Start Receiver
    receiver = mp.Process(target=receiver_worker, args=(queue, stop_event))
    receiver.start()
    
    # Start GPU Emitters
    emitters = []
    for i in range(num_gpus):
        p = mp.Process(target=gpu_worker, args=(i, queue, stop_event))
        p.start()
        emitters.append(p)
        
    # Run for the specified duration
    print(f"\n>>> Initiating Grokking Sequence for {DURATION_SECONDS} seconds... <<<\n")
    time.sleep(DURATION_SECONDS)
    
    # Signal shutdown
    print("\n>>> Halting Sequence... <<<")
    stop_event.set()
    
    for p in emitters:
        p.join()
    receiver.join()
    print("Experiment Complete.")

In [ ]:
# --- ANALYSIS & VISUALIZATION (The Moiré Effect) ---
print("Loading temporal data for Moiré analysis...")
data = np.load("time_dilation_data.npy", allow_pickle=True)

if len(data) > 0:
    # Extract the first packet's GPU clock data
    gpu_id, host_time, clock_data = data[0]
    
    # Calculate the delta (jitter) between consecutive clock cycles for the first 100 threads
    # Reshape based on samples_per_thread (50 in our config)
    samples_per_thread = 50
    threads_to_visualize = 100
    
    try:
        clock_matrix = clock_data[:threads_to_visualize * samples_per_thread].reshape(threads_to_visualize, samples_per_thread)
        
        # Calculate first-order differences (the jitter/dilation)
        jitter_matrix = np.diff(clock_matrix, axis=1)
        
        plt.figure(figsize=(12, 6))
        plt.imshow(jitter_matrix, aspect='auto', cmap='magma', interpolation='nearest')
        plt.colorbar(label='Clock Cycle Delta (Jitter)')
        plt.title('Quantum Jitter / Time Dilation Moiré Pattern (First 100 Threads)')
        plt.xlabel('Sample Sequence')
        plt.ylabel('Thread ID')
        plt.show()
        
        print("Visualization complete. The variations in color represent the 'Signature of Choice' and temporal jitter.")
    except Exception as e:
        print(f"Could not visualize data: {e}")
else:
    print("No data collected.")